In [ ]:
# ==========================================
# INTERVIEW LIVE DEMO SCRIPT
# ==========================================

# 1. Install bare minimum requirements
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

# 2. Mount your "Cloud Storage" (Google Drive)
from google.colab import drive
drive.mount('/content/drive')

from unsloth import FastLanguageModel
import torch

# 3. Load the DPO Adapters directly from Drive
print("\n[INFO] Loading fine-tuned DPO model...")
project_path = '/content/drive/MyDrive/Enterprise_SQL_LLM'
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = f"{project_path}/lora_dpo_final_model",
    max_seq_length = 2048,
    load_in_4bit = True, # 4-bit quantization prevents OOM
)
FastLanguageModel.for_inference(model) # Enable 2x faster inference

# 4. Define a Live Test Case
test_schema = """
CREATE TABLE server_logs (
    log_id INT,
    cpu_utilization FLOAT,
    timestamp DATETIME
);
"""
test_question = "Find the timestamp of the absolute lowest cpu utilization recorded."

messages = [
    {"role": "system", "content": "You are an expert SQL assistant. Your only job is to write exact, valid SQL queries based strictly on the provided database schema. Do not explain the query or include conversational filler. Return only the SQL query inside a markdown code block."},
    {"role": "user", "content": f"Database Schema:\n{test_schema}\n\nQuestion:\n{test_question}"}
]

# 5. Generate and Print
print("\n[INFO] Generating SQL...")
prompt = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_dict=True, return_tensors="pt").to("cuda")
outputs = model.generate(**prompt, max_new_tokens=256, use_cache=True, max_length=None, do_sample=False)
print("\n" + "="*40 + "\n MODEL RESPONSE \n" + "="*40)
print(tokenizer.decode(outputs[0][prompt["input_ids"].shape[1]:], skip_special_tokens=True))